# Increasing sample size experiments

Changes with sample size: spectral distributions, calibrated-selector sweeps, continuous ISE/energy minimizers, and the simultaneous sample/grid/scale limit.

Run the cells in order; all required functions are imported below. Every displayed figure is saved automatically under `outputs/pdfs/05_increasing_sample_size_experiments/`.

Source: [KDE chapter](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).


In [ ]:
from _setup import setup_notebook
from kde_tools.spectral_experiments import plot_discrete_bandwidth_distribution
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from kde_tools.bandwidth_histograms import plot_thesis_bandwidth_distributions
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.continuous_experiments import (
    draw_histograms,
    run_sample_size,
    save_continuous_distributions,
    style_panel,
    validate_continuous_distributions,
)
from kde_tools.continuous_minimizers import CONTINUOUS_SELECTOR_VERSION, METHODS, R_VALUES
from kde_tools.continuous_runs import CONTINUOUS_RUNNER_VERSION
from kde_tools.energy_minimization import (
    DEFAULT_ENERGY_SEARCH,
    EnergySearchConfig,
    energy_search_settings,
)
from kde_tools.fourier.similarities import (
    DEFAULT_FOURIER_GRID_TAG,
    DEFAULT_FREQUENCY_CUTOFF,
    DEFAULT_FREQUENCY_INTERVALS,
    FOURIER_L2_METHOD_VERSION,
    beta2_divergence_matrix_fourier,
)
from kde_tools.joint_experiments import (
    asymptotic_c_beta,
    asymptotic_grid_size,
    exact_expected_q_minimizer,
    psi_beta,
    richardson_local_metric_from_divergence,
)
from kde_tools.perron import (
    PERRON_SOLVER_VERSION,
    perron_eigenpair_from_divergence,
    perron_peak_index,
)
from kde_tools.spectral_experiments import (
    plot_spectral_sample_size_overlay,
    run_beta2_n_sweep_point,
)
from kde_tools.spectral_sweeps import SPECTRAL_SWEEP_RUNNER_VERSION, run_l2_point
from kde_tools.thesis_exports import save_minimizer_panel
from matplotlib.lines import Line2D
from threadpoolctl import threadpool_limits

paths = setup_notebook("05_increasing_sample_size_experiments")
code_directory = paths.code
pdf_output_directory = paths.pdfs
seed = 20260818
project_root = paths.project
kde_thesis_figure_dir = code_directory / "outputs/images/thesis"
kde_thesis_figure_dir.mkdir(parents=True, exist_ok=True)
ENERGY_SEARCH = EnergySearchConfig()
SEED = 20260823
H_MIN, H_MAX = (0.01, 2.0)
BETA = 2.0
FREQUENCY_CUTOFF = DEFAULT_FREQUENCY_CUTOFF
FREQUENCY_INTERVALS = DEFAULT_FREQUENCY_INTERVALS
FFT_HALF_WIDTH = np.pi * FREQUENCY_INTERVALS / FREQUENCY_CUTOFF
FFT_SPATIAL_POINTS = 2 * FREQUENCY_INTERVALS
plt.style.use("default")


## Spectral bandwidth distributions across sample sizes

The selected panels use n=300,400,500,1000 and retain the thesis delivery paths.
The all-size spectral overlays compare lambda=1 with lambda=N^(3/2), as in the chapter.


In [ ]:
joint_spectral_cache_root = code_directory / "outputs/data/shared_spectral_sweeps"


In [ ]:
beta2_n_sweep_values = np.array([300, 400, 500, 1000], dtype=int)
beta2_n_sweep_N = 1000
beta2_n_sweep_binning = 5
beta2_n_sweep_workers = 4
beta2_n_sweep_M = 1000
beta2_n_sweep_h_min = H_MIN
beta2_n_sweep_h_max = H_MAX
beta2_n_sweep_lambda = 1.0
beta2_n_sweep_localized_lambda = beta2_n_sweep_N**1.5
beta2_n_sweep_lambda_values = [beta2_n_sweep_lambda, beta2_n_sweep_localized_lambda]
beta2_n_sweep_seed = seed + 12
beta2_n_sweep_reuse_checkpoints = True
beta2_n_sweep_output_root = (
    code_directory / "outputs/data/spectral_scale_selection_kde/sample_size_sweep"
)
beta2_n_sweep_cache_root = code_directory / "outputs/data/shared_spectral_sweeps"
beta2_n_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_n_sweep_h_min, h_max=beta2_n_sweep_h_max, beta=2.0, grid_size=beta2_n_sweep_N
)
beta2_n_sweep_grid_indices = np.arange(beta2_n_sweep_N, dtype=float)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_oracle_indices = {
    int(current_n): float(
        np.interp(
            np.log(beta2_n_sweep_oracle_bandwidths[int(current_n)]),
            np.log(beta2_n_sweep_h_grid),
            beta2_n_sweep_grid_indices,
        )
    )
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_density_half_width = 12.0
beta2_n_sweep_density_spatial_points = 8192
beta2_n_sweep_dx = 2.0 * beta2_n_sweep_density_half_width / beta2_n_sweep_density_spatial_points
beta2_n_sweep_spatial_grid = -beta2_n_sweep_density_half_width + beta2_n_sweep_dx * np.arange(
    beta2_n_sweep_density_spatial_points
)
beta2_n_sweep_true_density = np.exp(-0.5 * beta2_n_sweep_spatial_grid**2) / np.sqrt(2.0 * np.pi)
beta2_n_sweep_frequencies = (
    2.0 * np.pi * np.fft.rfftfreq(beta2_n_sweep_density_spatial_points, d=beta2_n_sweep_dx)
)
beta2_n_sweep_parseval_weights = np.full(len(beta2_n_sweep_frequencies), 2.0)
beta2_n_sweep_parseval_weights[0] = 1.0
beta2_n_sweep_parseval_weights[-1] = 1.0
beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_settings = {
    "repetitions": beta2_n_sweep_M,
    "grid_size": beta2_n_sweep_N,
    "sigma2": 1.0,
    "h_min": beta2_n_sweep_h_min,
    "h_max": beta2_n_sweep_h_max,
    "fft_half_width": FFT_HALF_WIDTH,
    "fft_spatial_points": FFT_SPATIAL_POINTS,
    "seed": beta2_n_sweep_seed,
    "maximum_sample_size": int(np.max(beta2_n_sweep_values)),
}
beta2_n_sweep_configuration = {
    "settings": beta2_n_sweep_settings,
    "sample_sizes": [int(n) for n in beta2_n_sweep_values],
    "lambda_values": beta2_n_sweep_lambda_values,
    "spectral_sweep_runner": SPECTRAL_SWEEP_RUNNER_VERSION,
    "l2_method": FOURIER_L2_METHOD_VERSION,
    "perron_solver": PERRON_SOLVER_VERSION,
    "density_half_width": beta2_n_sweep_density_half_width,
    "density_spatial_points": beta2_n_sweep_density_spatial_points,
}
beta2_n_sweep_configuration_id = hashlib.sha256(
    json.dumps(beta2_n_sweep_configuration, sort_keys=True).encode()
).hexdigest()[:12]
beta2_n_sweep_output_directory = beta2_n_sweep_output_root / beta2_n_sweep_configuration_id
beta2_n_sweep_import_directories = [
    directory
    for profile in ("thesis", "preview")
    for directory in (
        code_directory / "outputs/data/fast_l2_bandwidth_distributions_vs_n_lambda" / profile
    ).glob("*")
    if directory.is_dir()
]
beta2_n_sweep_spectral_results = {}
beta2_n_sweep_results = {}
with threadpool_limits(limits=1):
    with ThreadPoolExecutor(max_workers=beta2_n_sweep_workers) as executor:
        for current_n, spectral_result, result in executor.map(
            partial(
                run_beta2_n_sweep_point,
                beta2_n_sweep_M=beta2_n_sweep_M,
                beta2_n_sweep_N=beta2_n_sweep_N,
                beta2_n_sweep_cache_root=beta2_n_sweep_cache_root,
                beta2_n_sweep_density_half_width=beta2_n_sweep_density_half_width,
                beta2_n_sweep_density_spatial_points=beta2_n_sweep_density_spatial_points,
                beta2_n_sweep_frequencies=beta2_n_sweep_frequencies,
                beta2_n_sweep_full_samples=beta2_n_sweep_full_samples,
                beta2_n_sweep_h_grid=beta2_n_sweep_h_grid,
                beta2_n_sweep_import_directories=beta2_n_sweep_import_directories,
                beta2_n_sweep_lambda_values=beta2_n_sweep_lambda_values,
                beta2_n_sweep_oracle_bandwidths=beta2_n_sweep_oracle_bandwidths,
                beta2_n_sweep_oracle_indices=beta2_n_sweep_oracle_indices,
                beta2_n_sweep_output_directory=beta2_n_sweep_output_directory,
                beta2_n_sweep_output_root=beta2_n_sweep_output_root,
                beta2_n_sweep_parseval_weights=beta2_n_sweep_parseval_weights,
                beta2_n_sweep_reuse_checkpoints=beta2_n_sweep_reuse_checkpoints,
                beta2_n_sweep_settings=beta2_n_sweep_settings,
                beta2_n_sweep_true_density=beta2_n_sweep_true_density,
            ),
            beta2_n_sweep_values,
        ):
            beta2_n_sweep_spectral_results[current_n] = spectral_result
            beta2_n_sweep_results[current_n] = result


In [ ]:
beta2_n_individual_figures = {}
for current_n in beta2_n_sweep_values:
    current_n = int(current_n)
    result = beta2_n_sweep_results[current_n]
    counts = np.stack([result[key] for key in ("counts", "ise_counts", "qhat_counts")])
    current_figure, _, bin_values = plot_thesis_bandwidth_distributions(
        beta2_n_sweep_h_grid,
        counts,
        sample_size=current_n,
        repetitions=beta2_n_sweep_M,
        positions_per_bin=beta2_n_sweep_binning,
    )
    beta2_n_individual_figures[current_n] = current_figure
    if current_n in {300, 400, 500, 1000}:
        current_figure.savefig(
            kde_thesis_figure_dir / f"distribution_of_chosen_bandwidths_n{current_n}.pdf",
            format="pdf",
            bbox_inches="tight",
        )
    starts, stops, grouped = bin_values
    np.savetxt(
        beta2_n_sweep_output_directory / f"n{current_n}" / "grouped_histogram_counts.csv",
        np.column_stack((starts, stops - 1, grouped.T)),
        delimiter=",",
        fmt="%d",
        header="first_grid_index,last_grid_index,spectral_lambda1,ise,qhat",
        comments="",
    )
    plt.show()
    plt.close(current_figure)
localized_overlay_plot = plot_spectral_sample_size_overlay(
    count_key="localized_counts",
    plot_title="Localized spectral bandwidth distributions across sample sizes",
    lambda_text="\\lambda_N=N^{3/2}",
    output_filename="localized_spectral_bandwidths_across_n_N1000.pdf",
    csv_filename="localized_overlay_histogram_counts.csv",
    tick_step=0.01,
    beta2_n_sweep_M=beta2_n_sweep_M,
    beta2_n_sweep_N=beta2_n_sweep_N,
    beta2_n_sweep_binning=beta2_n_sweep_binning,
    beta2_n_sweep_h_grid=beta2_n_sweep_h_grid,
    beta2_n_sweep_output_directory=beta2_n_sweep_output_directory,
    beta2_n_sweep_results=beta2_n_sweep_results,
    project_root=project_root,
)
lambda1_overlay_plot = plot_spectral_sample_size_overlay(
    count_key="counts",
    plot_title="Spectral bandwidth distributions across sample sizes",
    lambda_text="\\lambda=1",
    output_filename="spectral_bandwidths_across_n_lambda1_N1000.pdf",
    csv_filename="lambda1_overlay_histogram_counts.csv",
    tick_step=0.05,
    beta2_n_sweep_M=beta2_n_sweep_M,
    beta2_n_sweep_N=beta2_n_sweep_N,
    beta2_n_sweep_binning=beta2_n_sweep_binning,
    beta2_n_sweep_h_grid=beta2_n_sweep_h_grid,
    beta2_n_sweep_output_directory=beta2_n_sweep_output_directory,
    beta2_n_sweep_results=beta2_n_sweep_results,
    project_root=project_root,
)


## Sampling distributions for increasing sample size

We keep the calibrated numerical parameters $(N,\lambda)=(1000,1000)$ and the number of repetitions $M=1000$ fixed, while increasing the sample size through $n=100,200,\ldots,1000$. The samples are coupled across sample sizes: repetition $m$ uses the first $n$ observations of one common sample of length 1000. Each selection table is saved immediately after its sample size has finished.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
SAMPLE_SIZE_SWEEP_VALUES = np.arange(100, 1001, 100, dtype=int)
SAMPLE_SIZE_SWEEP_M = 1000
SAMPLE_SIZE_SWEEP_N = 1000
SAMPLE_SIZE_SWEEP_LAMBDA = 1000.0
SAMPLE_SIZE_SWEEP_SEED = 20260904
sample_size_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=SAMPLE_SIZE_SWEEP_N
)
sample_size_sweep_rng = np.random.default_rng(SAMPLE_SIZE_SWEEP_SEED)
sample_size_sweep_full_samples = sample_size_sweep_rng.normal(
    loc=0.0, scale=1.0, size=(SAMPLE_SIZE_SWEEP_M, int(SAMPLE_SIZE_SWEEP_VALUES.max()))
)
sample_size_sweep_data_directory = (
    pdf_output_directory.parents[1]
    / "data"
    / "experiment_1_joint_N_lambda_limit"
    / PERRON_SOLVER_VERSION
    / FOURIER_L2_METHOD_VERSION
    / DEFAULT_FOURIER_GRID_TAG
    / "sample_size_sweep"
)
sample_size_sweep_data_directory.mkdir(parents=True, exist_ok=True)
sample_size_sweep_results = {}
sample_size_sweep_summary_rows = []
for current_n in SAMPLE_SIZE_SWEEP_VALUES:
    current_n = int(current_n)
    current_start_time = time.perf_counter()
    current_settings = {
        "repetitions": SAMPLE_SIZE_SWEEP_M,
        "maximum_sample_size": int(SAMPLE_SIZE_SWEEP_VALUES.max()),
        "seed": SAMPLE_SIZE_SWEEP_SEED,
        "sigma2": 1.0,
        "fft_half_width": FFT_HALF_WIDTH,
        "fft_spatial_points": FFT_SPATIAL_POINTS,
    }
    current_spectral_result = run_l2_point(
        current_n,
        sample_size_sweep_full_samples,
        sample_size_sweep_h_grid,
        current_settings,
        [SAMPLE_SIZE_SWEEP_LAMBDA],
        sample_size_sweep_data_directory
        / "shared_runner"
        / f"N{SAMPLE_SIZE_SWEEP_N}_M{SAMPLE_SIZE_SWEEP_M}_seed{SAMPLE_SIZE_SWEEP_SEED}",
        cache_root=joint_spectral_cache_root,
        diagnostics=True,
    )
    selected_indices = current_spectral_result["selected_indices"][:, 0]
    solver_matvecs = current_spectral_result["solver_diagnostics"][:, 0, 2].astype(int)
    elapsed_seconds = time.perf_counter() - current_start_time
    selected_bandwidths = sample_size_sweep_h_grid[selected_indices]
    selection_counts = np.bincount(selected_indices, minlength=SAMPLE_SIZE_SWEEP_N)
    current_results = pd.DataFrame(
        {
            "repetition": np.arange(1, SAMPLE_SIZE_SWEEP_M + 1),
            "selected_index": selected_indices,
            "selected_h": selected_bandwidths,
            "solver_matvecs": solver_matvecs,
            "n": current_n,
            "M": SAMPLE_SIZE_SWEEP_M,
            "N": SAMPLE_SIZE_SWEEP_N,
            "lambda": SAMPLE_SIZE_SWEEP_LAMBDA,
            "seed": SAMPLE_SIZE_SWEEP_SEED,
            "beta": BETA,
            "h_min": H_MIN,
            "h_max": H_MAX,
            "fft_half_width": FFT_HALF_WIDTH,
            "fft_spatial_points": FFT_SPATIAL_POINTS,
            "l2_method": FOURIER_L2_METHOD_VERSION,
            "perron_solver": PERRON_SOLVER_VERSION,
        }
    )
    current_data_path = (
        sample_size_sweep_data_directory
        / f"selections_n{current_n}_N{SAMPLE_SIZE_SWEEP_N}_lambda{SAMPLE_SIZE_SWEEP_LAMBDA:g}_M{SAMPLE_SIZE_SWEEP_M}.csv"
    )
    current_results.to_csv(current_data_path, index=False)
    sample_size_sweep_results[current_n] = {
        "selected_indices": selected_indices.copy(),
        "selected_bandwidths": selected_bandwidths.copy(),
        "counts": selection_counts,
        "data_path": current_data_path,
    }
    sample_size_sweep_summary_rows.append(
        {
            "n": current_n,
            "M": SAMPLE_SIZE_SWEEP_M,
            "N": SAMPLE_SIZE_SWEEP_N,
            "lambda": SAMPLE_SIZE_SWEEP_LAMBDA,
            "mean selected h": selected_bandwidths.mean(),
            "median selected h": np.median(selected_bandwidths),
            "sd selected h": selected_bandwidths.std(ddof=1),
            "5% quantile": np.quantile(selected_bandwidths, 0.05),
            "95% quantile": np.quantile(selected_bandwidths, 0.95),
            "mean solver matvecs": solver_matvecs.mean(),
            "elapsed seconds": elapsed_seconds,
        }
    )
    print(f"Saved n={current_n} data: {current_data_path}")
sample_size_sweep_summary = pd.DataFrame(sample_size_sweep_summary_rows)
sample_size_sweep_summary_path = sample_size_sweep_data_directory / "summary.csv"
sample_size_sweep_summary.to_csv(sample_size_sweep_summary_path, index=False)
display(sample_size_sweep_summary.round(6))
print(f"Saved sample-size summary: {sample_size_sweep_summary_path}")


In [ ]:
sample_size_sweep_figures = {}
for current_n in SAMPLE_SIZE_SWEEP_VALUES:
    current_n = int(current_n)
    current_figure, current_axis = plot_discrete_bandwidth_distribution(
        sample_size_sweep_h_grid, sample_size_sweep_results[current_n]["counts"],
        title=f"Sampling distribution (n={current_n}, N={SAMPLE_SIZE_SWEEP_N}, lambda={SAMPLE_SIZE_SWEEP_LAMBDA:g})",
    )
    sample_size_sweep_figures[current_n] = current_figure
    plt.show()
    plt.close(current_figure)


## Continuous ISE and energy minimizers across sample sizes


For each $n=100,200,\ldots,2000$, compare the sampling distributions of four continuously optimized bandwidths on the common closed interval $h\in[0.01,2]$:

| Selector | Sample criterion |
|---|---|
| ISE | $\int(\widehat f_h-f)^2$ against the known $f=N(0,1)$ |
| $\widehat Q_0$ | $\tfrac12h^2\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_1$ | $\tfrac12\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_{13/14}$ | $\tfrac12h^{1/7}\|\partial_h\widehat f_h\|_2^2$ |

These subscripts use the thesis's **r parameter**, with
$\widehat Q_r(h)=\tfrac12h^{2(1-r)}\|\partial_h\widehat f_h\|_2^2$.
The ISE selector is the sample-specific oracle available here because the simulation density is known. Every criterion is evaluated with analytic Gaussian formulas. The search refines interior candidates continuously and compares both endpoints. Endpoint selections are part of the reported distributions.

The default is **M=1000 paired repetitions at each sample size**. The original 1000-by-1000 sample bank from seed `20260830` is retained exactly. A second block of 1000 observations per repetition is generated independently with `SeedSequence([20260830, 1])` and appended. Thus every sample for n≤1000 is unchanged, and the larger samples extend those same repetitions. Pre-unification results have been removed; new searches use the shared runner. A preview uses the first 12 repetitions of the same complete bank.

Sources: [thesis ISE and `prop:main_thm` defining Q_r](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [exact Gaussian ISE helper](../kde_tools/density.py); [continuous selector implementation and derived log-h derivatives](../kde_tools/continuous_minimizers.py); [earlier paired-sample settings](../inputs/qhat_sampling.json).

Run all cells. A separate resumable checkpoint is saved for each n. The figures are saved automatically as PDFs and also as PNGs. Two pages each show the complete interval and the interior selections with separate bars for the probability at the upper endpoint h=2. All histogram frequencies use M as denominator.

Reusable functions: [`continuous_experiments.py`](../kde_tools/continuous_experiments.py).
Experiment settings and calls are below. Every displayed figure is saved automatically as a PDF.


In [ ]:
plt.style.use("default")
notebook_name = "continuous_ise_qhat_distributions_vs_n"
data_output_directory = code_directory / "outputs/data" / notebook_name
image_output_directory = code_directory / "outputs/images" / notebook_name


## Settings and paired Gaussian samples

The optimization interval is the same [0.01,2] used in the earlier experiment. `HISTOGRAM_POINTS` controls the common display bins only. The minimizers are continuously refined and can fall between bin centers. The three Q criteria are sample energies, optimized separately within each repetition.


In [ ]:
RUN_PROFILE = "thesis"
PROFILES = {"thesis": {"repetitions": 1000}, "preview": {"repetitions": 12}}
SAMPLE_SIZES = [100, 1000, 1500, 2000]
SEED = 20260830
SAMPLE_BANK_REPETITIONS = 1000
SAMPLE_BANK_MAXIMUM_N = 2000
SAMPLE_BANK_BLOCK_SIZE = 1000
H_MIN, H_MAX = (0.01, 2.0)
SIGMA2 = 1.0
SCAN_POINTS = DEFAULT_ENERGY_SEARCH.scan_points
SPATIAL_POINTS = DEFAULT_ENERGY_SEARCH.spatial_points
HALF_WIDTH = DEFAULT_ENERGY_SEARCH.half_width
HISTOGRAM_POINTS = 201
WORKERS = 6
REUSE_CHECKPOINTS = True
M = PROFILES[RUN_PROFILE]["repetitions"]
if not 1 <= M <= SAMPLE_BANK_REPETITIONS:
    raise ValueError("Repetitions must fit in the fixed paired sample bank.")
if not SAMPLE_SIZES or min(SAMPLE_SIZES) < 1 or max(SAMPLE_SIZES) > SAMPLE_BANK_MAXIMUM_N:
    raise ValueError("Sample sizes must fit in the fixed sample bank.")
sample_blocks = []
for block_index in range(
    (SAMPLE_BANK_MAXIMUM_N + SAMPLE_BANK_BLOCK_SIZE - 1) // SAMPLE_BANK_BLOCK_SIZE
):
    block_seed = SEED if block_index == 0 else np.random.SeedSequence([SEED, block_index])
    sample_blocks.append(
        np.random.default_rng(block_seed).normal(
            0.0, np.sqrt(SIGMA2), size=(SAMPLE_BANK_REPETITIONS, SAMPLE_BANK_BLOCK_SIZE)
        )
    )
full_sample_bank = np.concatenate(sample_blocks, axis=1)[:, :SAMPLE_BANK_MAXIMUM_N]
prefix_sample_bank_sha256 = hashlib.sha256(
    full_sample_bank[:, :SAMPLE_BANK_BLOCK_SIZE].tobytes()
).hexdigest()
full_samples = full_sample_bank[:M]
configuration = {
    "version": CONTINUOUS_SELECTOR_VERSION,
    "energy_search": energy_search_settings(),
    "runner": CONTINUOUS_RUNNER_VERSION,
    "methods": list(METHODS),
    "r_values": R_VALUES,
    "sample_sizes": SAMPLE_SIZES,
    "repetitions": M,
    "seed": SEED,
    "sigma2": SIGMA2,
    "h_min": H_MIN,
    "h_max": H_MAX,
    "sample_bank_shape": list(full_sample_bank.shape),
    "sample_bank_generation": "fixed_1000_column_blocks_preserving_original_prefix_v1",
    "sample_bank_block_size": SAMPLE_BANK_BLOCK_SIZE,
    "prefix_sample_bank_sha256": prefix_sample_bank_sha256,
    "sample_bank_sha256": hashlib.sha256(full_sample_bank.tobytes()).hexdigest(),
    "scan_points": SCAN_POINTS,
    "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH,
    "histogram_points": HISTOGRAM_POINTS,
    "source_sha256": {
        filename: hashlib.sha256((code_directory / "kde_tools" / filename).read_bytes()).hexdigest()
        for filename in ("continuous_minimizers.py", "qhat.py")
    },
}
configuration_json = json.dumps(configuration, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(json.dumps(configuration, indent=2) + "\n")
histogram_bandwidths = np.geomspace(H_MIN, H_MAX, HISTOGRAM_POINTS)
histogram_positions = np.arange(HISTOGRAM_POINTS, dtype=float)
histogram_edges = np.arange(HISTOGRAM_POINTS + 1) - 0.5
continuous_cache_root = code_directory / "outputs/data/shared_continuous_minimizers"
continuous_import_directories = []
print(f"M={M} paired repetitions; n={SAMPLE_SIZES}; continuous interval=[{H_MIN},{H_MAX}]")
print(f"Checkpoint and CSV directory: {run_directory}")


## Compute or resume all four continuous minimizers

The same sample supplies all four selectors. A Fourier scan supplies candidate basins, followed by analytic Gaussian-pair refinement and comparison with both endpoints. A finite scan is a numerical search, rather than a certificate that every stationary point was enumerated. Later validation doubles both scan and Fourier resolution on three repetitions at every n.


In [ ]:
search_settings = {
    "h_min": H_MIN,
    "h_max": H_MAX,
    "sigma2": SIGMA2,
    "scan_points": SCAN_POINTS,
    "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH,
}
results_by_n = {}
with threadpool_limits(limits=1):
    for current_n in SAMPLE_SIZES:
        results_by_n[current_n] = run_sample_size(
            current_n,
            reuse=REUSE_CHECKPOINTS,
            workers=WORKERS,
            configuration=configuration,
            continuous_cache_root=continuous_cache_root,
            continuous_import_directories=continuous_import_directories,
            full_samples=full_samples,
            run_directory=run_directory,
            search_settings=search_settings,
        )


## Saved distributions, boundary counts, and summaries

Each CSV selection row records a continuously optimized bandwidth and its numerical diagnostics. Histogram frequencies use the same geometric bin centers for all four selectors and divide by M. The summary explicitly counts selections at both endpoints. Interior-only frequencies below keep the same denominator M.

`peak_comparison.csv` records the ISE mode bin, the green interior mode bin, boundary mass, and paired bandwidth comparisons on samples where Q₁₃/₁₄ has an interior minimizer. These are descriptive histogram and sample comparisons.


In [ ]:
frequencies, boundary_masses = save_continuous_distributions(
    histogram_points=HISTOGRAM_POINTS,
    repetitions=M,
    sample_sizes=SAMPLE_SIZES,
    histogram_bandwidths=histogram_bandwidths,
    histogram_edges=histogram_edges,
    histogram_positions=histogram_positions,
    results_by_n=results_by_n,
    run_directory=run_directory,
)


## Numerical validation

Repeat the continuous searches at doubled scan and Fourier resolution on three paired repetitions at each n. Independently evaluate the exact ISE helper at all four selected bandwidths and both endpoints for these samples; the selected ISE must be no larger than any of those alternatives. Check stationarity and positive curvature for all interior solutions, plus the appropriate one-sided derivative signs at endpoints.


In [ ]:
validation_report = validate_continuous_distributions(
    half_width=HALF_WIDTH,
    h_max=H_MAX,
    h_min=H_MIN,
    repetitions=M,
    sample_sizes=SAMPLE_SIZES,
    scan_points=SCAN_POINTS,
    sigma2=SIGMA2,
    spatial_points=SPATIAL_POINTS,
    full_samples=full_samples,
    results_by_n=results_by_n,
    run_directory=run_directory,
)


## Distribution plots

The first figure shows all selections on the complete bandwidth interval. In the second, each main panel shows the histograms after removing only exact upper-endpoint selections; the inset bars show the probability at h=2. Main-panel frequencies still divide by all M repetitions. The main-panel range includes every occupied interior bin across all sample sizes and methods. Together, the interior histograms and inset bars account for the complete probability distributions.

The twenty sample sizes are split into two readable pages: n=100,…,1000 and n=1100,…,2000. Axis limits are shared across both pages.


In [ ]:
COLORS = {"ISE": "#7B4AB2", "Q0": "#0072B2", "Q1": "#D55E00", "Q13_14": "#009E73"}
LABELS = {
    "ISE": "ISE",
    "Q0": "$\\widehat Q_0$",
    "Q1": "$\\widehat Q_1$",
    "Q13_14": "$\\widehat Q_{13/14}$",
}
STYLES = {"ISE": "-", "Q0": "-", "Q1": "--", "Q13_14": "-."}
legend_handles = [
    Line2D([0], [0], color=COLORS[m], linestyle=STYLES[m], linewidth=2, label=LABELS[m])
    for m in METHODS
]
full_y_max = min(
    1.06, 1.06 * max((v.max() for point in frequencies.values() for v in point.values()))
)
full_ticks = np.unique(np.linspace(0, HISTOGRAM_POINTS - 1, 7, dtype=int))
for current_n in SAMPLE_SIZES:
    panel, panel_ax = plt.subplots(figsize=(4.2, 3.0), layout="constrained")
    draw_histograms(
        panel_ax,
        frequencies[current_n],
        colors=COLORS,
        styles=STYLES,
        histogram_edges=histogram_edges,
    )
    style_panel(
        panel_ax,
        current_n,
        full_ticks,
        full_y_max,
        HISTOGRAM_POINTS - 0.3,
        histogram_bandwidths=histogram_bandwidths,
    )
    boundary_ax = panel_ax.inset_axes([0.56, 0.48, 0.28, 0.42])
    boundary_ax.bar(
        np.arange(len(METHODS)),
        [boundary_masses[current_n][m] for m in METHODS],
        color=[COLORS[m] for m in METHODS],
        width=0.67,
    )
    boundary_ax.set_ylim(0, 1.05)
    boundary_ax.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
    boundary_ax.set_xticks([])
    boundary_ax.tick_params(axis="y", labelsize=8, length=2)
    boundary_ax.set_title(f"at $h={H_MAX:g}$", fontsize=9, pad=4)
    boundary_ax.patch.set_facecolor("#FAFBFD")
    for side in ("top", "right"):
        boundary_ax.spines[side].set_visible(False)
    panel_ax.set_title(f"$n={current_n}$; $M={M}$", fontsize=12, pad=8)
    panel.legend(
        handles=legend_handles, loc="outside lower center", ncol=4, frameon=False, fontsize=8
    )
    save_minimizer_panel(panel, current_n)
    plt.show()
    plt.close(panel)


## Simultaneous sample, grid, and scale limits

The original joint-notebook study with increasing sample size and a rescaled bandwidth window. Its beta setting is explicit below.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
ASYMPTOTIC_BETA = 1 / 7
if not -2.0 < ASYMPTOTIC_BETA < 3.0:
    raise ValueError("The interior asymptotic minimizer requires -2 < beta < 3.")
ASYMPTOTIC_M = 200
ASYMPTOTIC_N_VALUES = np.array([100, 200, 400, 800, 1600, 3200], dtype=int)
ASYMPTOTIC_SEED = 20260906
ASYMPTOTIC_T_MIN = 0.35
ASYMPTOTIC_T_MAX = 2.2
ASYMPTOTIC_GRID_BASE = 250
ASYMPTOTIC_GRID_GAMMA = 0.25
ASYMPTOTIC_ALPHA = 1.0
ASYMPTOTIC_C_BETA = asymptotic_c_beta(ASYMPTOTIC_BETA)
print("Theoretical asymptotic constant:", ASYMPTOTIC_C_BETA)
if not ASYMPTOTIC_T_MIN < ASYMPTOTIC_C_BETA < ASYMPTOTIC_T_MAX:
    raise ValueError("The theoretical c_beta must lie inside the chosen scaled t-window.")
asymptotic_rng = np.random.default_rng(ASYMPTOTIC_SEED)
asymptotic_full_samples = asymptotic_rng.normal(
    loc=0.0, scale=1.0, size=(ASYMPTOTIC_M, int(ASYMPTOTIC_N_VALUES.max()))
)
asymptotic_rows = []
local_metric_validation = None
asymptotic_start_time = time.perf_counter()
for current_n in ASYMPTOTIC_N_VALUES:
    current_n = int(current_n)
    current_N = asymptotic_grid_size(
        current_n,
        base_size=ASYMPTOTIC_GRID_BASE,
        gamma=ASYMPTOTIC_GRID_GAMMA,
        sample_sizes=ASYMPTOTIC_N_VALUES,
    )
    current_t_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=ASYMPTOTIC_T_MIN, h_max=ASYMPTOTIC_T_MAX, beta=ASYMPTOTIC_BETA, grid_size=current_N
    )
    current_scale = current_n ** (-0.2)
    current_h_grid = current_scale * current_t_grid
    current_psi_grid = np.array([psi_beta(h, ASYMPTOTIC_BETA) for h in current_h_grid])
    current_delta_psi = float(current_psi_grid[1] - current_psi_grid[0])
    current_lambda = float(current_n**0.8 * current_N**ASYMPTOTIC_ALPHA)
    current_effective_lambda = float(current_lambda * current_n ** (-0.8))
    current_h_expected = exact_expected_q_minimizer(
        current_n, ASYMPTOTIC_BETA, ASYMPTOTIC_T_MIN, ASYMPTOTIC_T_MAX, search=ENERGY_SEARCH
    )
    current_t_expected = float(current_n**0.2 * current_h_expected)
    current_start_time = time.perf_counter()
    for repetition in range(ASYMPTOTIC_M):
        current_samples = asymptotic_full_samples[repetition, :current_n]
        current_divergence = beta2_divergence_matrix_fourier(
            current_samples,
            current_h_grid,
            half_width=FFT_HALF_WIDTH,
            spatial_points=FFT_SPATIAL_POINTS,
        )
        current_divergence = np.maximum(current_divergence, 0.0)
        np.fill_diagonal(current_divergence, 0.0)
        perron_result = perron_eigenpair_from_divergence(current_divergence, current_lambda)
        current_perron_vector = perron_result.vector
        matvec_count = perron_result.matvecs
        current_perron_index = int(perron_peak_index(current_perron_vector))
        current_h_perron = float(current_h_grid[current_perron_index])
        current_q_local = richardson_local_metric_from_divergence(
            current_divergence, current_delta_psi
        )
        finite_q_indices = np.flatnonzero(np.isfinite(current_q_local))
        if finite_q_indices.size == 0:
            raise RuntimeError("No finite positive local-metric estimates.")
        current_q_index = int(finite_q_indices[np.argmin(current_q_local[finite_q_indices])])
        current_h_q = float(current_h_grid[current_q_index])
        current_t_perron = float(current_n**0.2 * current_h_perron)
        current_t_q = float(current_n**0.2 * current_h_q)
        asymptotic_rows.append(
            {
                "n": current_n,
                "repetition": repetition + 1,
                "N": current_N,
                "lambda": current_lambda,
                "effective_lambda": current_effective_lambda,
                "effective_lambda_over_N2": current_effective_lambda / current_N**2,
                "beta": ASYMPTOTIC_BETA,
                "h_perron": current_h_perron,
                "h_q": current_h_q,
                "h_expected_q": current_h_expected,
                "t_perron": current_t_perron,
                "t_q": current_t_q,
                "t_expected_q": current_t_expected,
                "c_beta": ASYMPTOTIC_C_BETA,
                "scaled_spectral_gap": abs(current_t_perron - current_t_q),
                "solver_matvecs": matvec_count,
            }
        )
        if (repetition + 1) % 25 == 0 or repetition + 1 == ASYMPTOTIC_M:
            print(f"n={current_n}, N={current_N}: {repetition + 1}/{ASYMPTOTIC_M} repetitions")
    current_elapsed = time.perf_counter() - current_start_time
    print(
        f"Finished n={current_n}, N={current_N}, lambda={current_lambda:.4g} in {current_elapsed:.1f} s."
    )
asymptotic_results = pd.DataFrame(asymptotic_rows)
asymptotic_elapsed = time.perf_counter() - asymptotic_start_time
print(f"Completed full asymptotic experiment in {asymptotic_elapsed:.1f} s.")
asymptotic_summary = asymptotic_results.groupby("n", as_index=False).agg(
    N=("N", "first"),
    lambda_value=("lambda", "first"),
    effective_lambda=("effective_lambda", "first"),
    effective_lambda_over_N2=("effective_lambda_over_N2", "first"),
    mean_t_perron=("t_perron", "mean"),
    median_t_perron=("t_perron", "median"),
    q10_t_perron=("t_perron", lambda x: np.quantile(x, 0.1)),
    q90_t_perron=("t_perron", lambda x: np.quantile(x, 0.9)),
    mean_t_q=("t_q", "mean"),
    median_t_q=("t_q", "median"),
    q10_t_q=("t_q", lambda x: np.quantile(x, 0.1)),
    q90_t_q=("t_q", lambda x: np.quantile(x, 0.9)),
    t_expected_q=("t_expected_q", "first"),
    c_beta=("c_beta", "first"),
    mean_scaled_spectral_gap=("scaled_spectral_gap", "mean"),
    median_scaled_spectral_gap=("scaled_spectral_gap", "median"),
)
display(asymptotic_summary.round(8))
fig, ax = plt.subplots(figsize=(9.8, 6.0))
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_t_perron"],
    marker="o",
    linewidth=2.2,
    label="MC mean of $n^{1/5}h_{\\rm Perron}$",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_t_q"],
    marker="s",
    linewidth=2.0,
    label="MC mean of $n^{1/5}\\arg\\min\\widehat Q$",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["t_expected_q"],
    marker="^",
    linestyle="--",
    linewidth=1.9,
    label="$n^{1/5}\\arg\\min E[\\widehat Q]$",
)
ax.axhline(
    ASYMPTOTIC_C_BETA, linestyle=":", linewidth=2.0, label=f"$c_\\beta={ASYMPTOTIC_C_BETA:.4f}$"
)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="rescaled bandwidth $n^{1/5}h$",
    title=f"Full Perron-to-asymptotic bridge, $\\beta={ASYMPTOTIC_BETA:g}$",
)
ax.legend()
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["mean_scaled_spectral_gap"],
    marker="o",
    linewidth=2.2,
    label="Monte Carlo mean",
)
ax.plot(
    asymptotic_summary["n"],
    asymptotic_summary["median_scaled_spectral_gap"],
    marker="s",
    linestyle="--",
    linewidth=1.8,
    label="Monte Carlo median",
)
ax.axhline(0.0, linestyle=":", linewidth=1.5)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="$n^{1/5}|h_{\\rm Perron}-\\hat h_Q|$",
    title="Spectral error on the statistical bandwidth scale",
)
ax.legend()
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9.5, 5.8))
n_values_plot = asymptotic_summary["n"].to_numpy(dtype=float)
ax.fill_between(
    n_values_plot,
    asymptotic_summary["q10_t_perron"].to_numpy(dtype=float),
    asymptotic_summary["q90_t_perron"].to_numpy(dtype=float),
    alpha=0.2,
    label="10%-90% Monte Carlo band",
)
ax.plot(
    n_values_plot,
    asymptotic_summary["median_t_perron"],
    marker="o",
    linewidth=2.2,
    label="median $n^{1/5}h_{\\rm Perron}$",
)
ax.axhline(
    ASYMPTOTIC_C_BETA, linestyle=":", linewidth=2.0, label=f"$c_\\beta={ASYMPTOTIC_C_BETA:.4f}$"
)
ax.set(
    xscale="log",
    xlabel="sample size $n$",
    ylabel="$n^{1/5}h_{\\rm Perron}$",
    title="Sampling concentration of the rescaled Perron peak",
)
ax.legend()
fig.tight_layout()
plt.show()
